# M11.11 · AlphaGo'dan AlphaZero'ya — politika/değer ağları ve kendine karşı oyunla MCTS

AlphaGo'nun dört aşaması (uzman hamleleriyle denetimli politika ağı, kendine karşı oyunda REINFORCE, değer ağı regresyonu, politika ve değer ağlarıyla PUCT araması) ve AlphaGo Zero'nun sadeleştirmesi: tek ağ $(p,v)=f_\theta(s)$, yalnız kendine karşı oyun, MCTS ziyaret sayılarından politika hedefi, $(z-v)^2-\pi^\top\log p+c\|\theta\|^2$ kaybı, kökte Dirichlet gürültüsü, sıcaklık ve simetri artırımı. Saf Python'da PUCT, NumPy'da 8 simetri ve Elo/Bradley–Terry, PyTorch'ta tic-tac-toe'yu sıfırdan öğrenip kusursuz oyuncuyla berabere kalan mini AlphaZero, sonda yüzlerce oyunu aynı anda oynatan toplu yaprak değerlendirmeli kendine karşı oyun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-11-alphazero.html](https://m-gonul.github.io/pytorch-lab/m11-11-alphazero.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_11_alphazero.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\nLN = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]\nMEMO = {}\n\n\ndef term(b):\n    for l in LN:\n        s = b[l[0]] + b[l[1]] + b[l[2]]\n        if abs(s) == 3:\n            return s // 3\n    return 0 if all(b) else None\n\n\ndef perfect(b, p):\n    k = (tuple(b), p)\n    if k in MEMO:\n        return MEMO[k]\n    t = term(b)\n    if t is not None:\n        return t, []\n    vals = {}\n    for m in range(9):\n        if b[m] == 0:\n            b2 = b[:]\n            b2[m] = p\n            vals[m] = perfect(b2, -p)[0]\n    best = max(v * p for v in vals.values()) * p\n    MEMO[k] = (best, [m for m, v in vals.items() if v == best])\n    return MEMO[k]\n\n\ndef oracle(board, player):\n    return [1 / 9] * 9, float(perfect(list(board), player)[0] * player)\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def vp():\n        close(need(ns, \"visit_policy\")([10, 30, 60], 0.5)[2], 3600 / 4600, what=\"τ = 0.5\")\n        ensure(ns[\"visit_policy\"]([3, 7, 7], 0) == [0.0, 1.0, 0.0], \"τ = 0: argmax\")\n        ensure(need(ns, \"terminal\")([1, 1, 1, 0, -1, -1, 0, 0, 0]) == 1 and ns[\"terminal\"]([0] * 9) is None, \"terminal\")\n    R_.test(\"visit_policy ve terminal\", vp)\n\n    def search():\n        u = lambda b, p: ([1 / 9] * 9, 0.0)\n        N = need(ns, \"puct_search\")([1, 1, 0, -1, -1, 0, 0, 0, 0], 1, u, 60)\n        ensure(max(range(9), key=lambda i: N[i]) == 2, \"anında kazanç bulunmalı\")\n        N = ns[\"puct_search\"]([1, 1, 0, -1, 0, 0, 0, 0, 0], -1, u, 200)\n        ensure(max(range(9), key=lambda i: N[i]) == 2, \"rakibin kazancı engellenmeli\")\n        import random\n        rng = random.Random(0)\n        rows, ok = [], 0\n        positions = []\n        while len(positions) < 30:\n            b, p = [0] * 9, 1\n            for _ in range(rng.randint(0, 5)):\n                if term(b) is not None:\n                    break\n                m = rng.choice([i for i in range(9) if b[i] == 0])\n                b[m] = p\n                p = -p\n            if term(b) is None:\n                positions.append((b, p))\n        okz = 0\n        for b, p in positions:\n            Nn = ns[\"puct_search\"](b, p, oracle, 100)\n            ok += max(range(9), key=lambda i: Nn[i]) in perfect(b, p)[1]\n            Nz = ns[\"puct_search\"](b, p, u, 100)\n            okz += max(range(9), key=lambda i: Nz[i]) in perfect(b, p)[1]\n        ensure(ok >= 29, f\"kusursuz değerle arama optimal hamleyi seçmeli ({ok}/30)\")\n        R_.table(\"30 rastgele konum, 100 benzetim\", [\"ağ\", \"optimal hamle seçimi\"], [[\"önsel düzgün, değer = minimax (kusursuz)\", f\"{ok}/30\"], [\"önsel düzgün, değer = 0 (bilgisiz)\", f\"{okz}/30\"]],\n                 note=\"Bilgisiz değerle PUCT yalnız terminal düğümlerden bilgi alır: sığ taktikleri bulur, derin planı bulamaz. AlphaZero'da değer başlığının işi bu boşluğu doldurmaktır.\")\n    R_.test(\"puct_search\", search)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"döngü kuralları\", lambda: (max_loops(src, 0, func=\"augment\"), max_loops(src, 0, func=\"az_loss\"), max_loops(src, 1, func=\"fit_elo\")))\n\n    def sym():\n        P = need(ns, \"symmetry_perms\")()\n        ensure(P.shape == (8, 9) and len({tuple(p) for p in P}) == 8, \"8 farklı permütasyon\")\n        ensure(all(sorted(p) == list(range(9)) for p in P) and all(p[4] == 4 for p in P), \"permütasyon, merkez sabit\")\n        corners = {0, 2, 6, 8}\n        ensure(all({int(p[i]) for i in corners} == corners for p in P), \"köşeler köşelere\")\n        rng = np.random.default_rng(0)\n        X = rng.integers(0, 2, (5, 18)).astype(float)\n        PI = rng.dirichlet(np.ones(9), 5)\n        Xa, PIa, Za = need(ns, \"augment\")(X, PI, np.arange(5.0))\n        ensure(Xa.shape == (40, 18) and PIa.shape == (40, 9) and Za.shape == (40,), \"şekiller\")\n        close(np.sort(Xa[Za == 3][:, :9].sum(1)), np.full(8, X[3, :9].sum()), what=\"taş sayısı korunur\")\n        close(PIa.sum(1), np.ones(40), what=\"π hâlâ dağılım\")\n    R_.test(\"simetriler\", sym)\n\n    def loss():\n        v = need(ns, \"az_loss\")(np.array([1.0]), np.array([0.5]), np.array([[0.5, 0.5]]), np.array([[0.0, 0.0]]), 0.0)\n        close(v, 0.25 + np.log(2), what=\"el hesabı\")\n        big = ns[\"az_loss\"](np.array([1.0]), np.array([1.0]), np.array([[1.0, 0.0]]), np.array([[1000.0, 0.0]]), 0.0)\n        ensure(np.isfinite(big) and abs(big) < 1e-9, \"kararlı log-softmax\")\n    R_.test(\"az_loss\", loss)\n\n    def elo():\n        close(need(ns, \"elo_expected\")(1200, 1000), 1 / (1 + 10 ** -0.5), what=\"beklenen skor\")\n        rng = np.random.default_rng(1)\n        true = np.array([0.0, 150, 300, 600, 900])\n        K = len(true)\n        Wm = np.zeros((K, K))\n        for i in range(K):\n            for j in range(K):\n                if i != j:\n                    Wm[i, j] = rng.binomial(400, ns[\"elo_expected\"](true[i], true[j]))\n        Wm = (Wm + (400 - Wm.T)) / 2\n        np.fill_diagonal(Wm, 0)\n        r = need(ns, \"fit_elo\")(Wm)\n        close(r - r[0], true - true[0], atol=40, what=\"dereceler geri kazanılır\")\n        R_.table(\"Bradley–Terry ile Elo (her çift 400 oyun)\", [\"oyuncu\", \"gerçek\", \"kestirim\"], [[i, f\"{true[i] - true[0]:.0f}\", f\"{r[i] - r[0]:.0f}\"] for i in range(K)],\n                 note=\"Elo yalnız farklar için anlamlıdır; AlphaGo makaleleri bir oyuncuyu (ör. GnuGo = 0 ya da rastgele oyuncu) sabitleyerek ölçekler.\")\n    R_.test(\"Elo / Bradley–Terry\", elo)\n",
 "d": "import random\nimport numpy as np\nimport torch\nfrom labcheck import close, need, ensure\n\nMEMO = {}\n\n\ndef perfect(b, p, term):\n    k = (tuple(b), p)\n    if k in MEMO:\n        return MEMO[k]\n    t = term(b)\n    if t is not None:\n        return t, []\n    vals = {}\n    for m in range(9):\n        if b[m] == 0:\n            b2 = b[:]\n            b2[m] = p\n            vals[m] = perfect(b2, -p, term)[0]\n    best = max(v * p for v in vals.values()) * p\n    MEMO[k] = (best, [m for m, v in vals.items() if v == best])\n    return MEMO[k]\n\n\ndef states(term):\n    out, stack = set(), [([0] * 9, 1)]\n    while stack:\n        b, p = stack.pop()\n        if (tuple(b), p) in out or term(b) is not None:\n            continue\n        out.add((tuple(b), p))\n        for m in range(9):\n            if b[m] == 0:\n                b2 = b[:]\n                b2[m] = p\n                stack.append((b2, -p))\n    return out\n\n\ndef policy_acc(net, ns, S):\n    ok = 0\n    for b, p in S:\n        with torch.no_grad():\n            lo = ns[\"Net\"].forward(net, ns[\"encode\"](list(b), p)[None])[0][0].numpy()\n        lo[np.array(b) != 0] = -1e9\n        ok += int(lo.argmax()) in perfect(list(b), p, ns[\"term\"])[1]\n    return ok / len(S)\n\n\ndef play(ns, net, opp, net_player, seed):\n    rng = np.random.default_rng(seed)\n    r = random.Random(seed)\n    b, pl = [0] * 9, 1\n    while ns[\"term\"](b) is None:\n        if pl == net_player:\n            a = int(ns[\"mcts\"](net, b, pl, 50, rng, noise=False).argmax())\n        elif opp == \"random\":\n            a = r.choice([i for i in range(9) if b[i] == 0])\n        else:\n            a = perfect(b, pl, ns[\"term\"])[1][0]\n        b[a] = pl\n        pl = -pl\n    return ns[\"term\"](b) * net_player\n\n\ndef check(ns, R_):\n    def parts():\n        net = need(ns, \"Net\")()\n        x = need(ns, \"encode\")([1, -1, 0, 0, 0, 0, 0, 0, 0], -1)\n        ensure(x.tolist()[:2] == [0.0, 1.0] and x.tolist()[9:11] == [1.0, 0.0], \"sıradaki oyuncunun bakışı\")\n        lo, v = net(x[None])\n        ensure(tuple(lo.shape) == (1, 9) and -1 <= float(v) <= 1, \"çıktılar\")\n        N = need(ns, \"mcts\")(net, [1, 1, 0, -1, -1, 0, 0, 0, 0], 1, 50, np.random.default_rng(0), noise=False)\n        ensure(int(np.argmax(N)) == 2 and N.sum() == 50, \"eğitimsiz ağla bile anında kazancı bulur\")\n        data = need(ns, \"self_play\")(net, 10, np.random.default_rng(1))\n        ensure(all(abs(z) <= 1 and abs(sum(pi) - 1) < 1e-6 for _, _, pi, z in data), \"örnekler\")\n        ensure(len({z * p for _, p, _, z in data}) == 1, \"z sıradaki oyuncunun bakışından tutarlı\")\n    R_.test(\"bileşenler\", parts)\n\n    def full():\n        S = states(ns[\"term\"])\n        torch.manual_seed(0)\n        acc0 = policy_acc(ns[\"Net\"](), ns, S)\n        net = need(ns, \"alphazero\")(15, 20, 30, 0)\n        acc = policy_acc(net, ns, S)\n        rp = [play(ns, net, \"perfect\", 1, s) for s in range(3)] + [play(ns, net, \"perfect\", -1, s) for s in range(3)]\n        rr = [play(ns, net, \"random\", 1, s) for s in range(5)] + [play(ns, net, \"random\", -1, s) for s in range(5)]\n        ensure(acc > 0.8 and acc > acc0 + 0.2, f\"politika başlığı optimal hamleleri öğrenmeli ({acc0:.2f} → {acc:.2f})\")\n        ensure(all(v >= 0 for v in rp), \"kusursuz oyuncuya karşı kaybetmemeli\")\n        ensure(all(v >= 0 for v in rr) and sum(v == 1 for v in rr) >= 7, \"rastgeleyi yenmeli\")\n        R_.table(\"Mini AlphaZero, 15 yineleme × 20 oyun × 30 benzetim\", [\"ölçü\", \"değer\"],\n                 [[f\"politika başlığı (aramasız) optimal hamle oranı, {len(S)} durum\", f\"{acc0:.2f} → {acc:.2f}\"],\n                  [\"MCTS(50) – kusursuz oyuncu (6 oyun, her iki renk)\", f\"{rp.count(1)}G {rp.count(0)}B {rp.count(-1)}M\"],\n                  [\"MCTS(50) – rastgele (10 oyun)\", f\"{rr.count(1)}G {rr.count(0)}B {rr.count(-1)}M\"]],\n                 note=\"İnsan verisi, rollout ve el yapımı değerlendirme yok: yalnız kurallar ve kendine karşı oyun. Ziyaret sayıları ağın önselinden daha güçlü bir politikadır; ağ onu taklit ederek iyileşir.\")\n    R_.test(\"alphazero\", full)\n",
 "e": "import math\nimport numpy as np\nimport torch\nfrom labcheck import need, ensure, bench, fmt_time\n\n\ndef seq_self_play(net, n_games, n_sims, rng, term, c=1.5):\n    out = []\n    for _ in range(n_games):\n        b, pl, hist = [0] * 9, 1, []\n        while term(b) is None:\n            root = {\"b\": b[:], \"pl\": pl}\n\n            def node(bb, p):\n                legal = np.array([x == 0 for x in bb])\n                with torch.no_grad():\n                    B = torch.tensor(bb, dtype=torch.float32) * p\n                    lo, v = net(torch.cat([(B == 1).float(), (B == -1).float()])[None])\n                pr = torch.softmax(lo[0], 0).numpy() * legal\n                return {\"b\": bb, \"pl\": p, \"legal\": legal, \"P\": pr / pr.sum(), \"v\": float(v[0]), \"N\": np.zeros(9), \"W\": np.zeros(9), \"ch\": {}}\n\n            root = node(b[:], pl)\n            eta = np.zeros(9)\n            eta[root[\"legal\"]] = rng.dirichlet([0.3] * int(root[\"legal\"].sum()))\n            root[\"P\"] = 0.75 * root[\"P\"] + 0.25 * eta\n            for _ in range(n_sims):\n                nd, path = root, []\n                while True:\n                    Q = np.where(nd[\"N\"] > 0, nd[\"W\"] / np.maximum(nd[\"N\"], 1), 0.0)\n                    U = c * nd[\"P\"] * math.sqrt(nd[\"N\"].sum() + 1) / (1 + nd[\"N\"])\n                    a = int(np.argmax(np.where(nd[\"legal\"], Q + U, -1e9)))\n                    path.append((nd, a))\n                    if a in nd[\"ch\"]:\n                        nd = nd[\"ch\"][a]\n                        continue\n                    b2 = nd[\"b\"][:]\n                    b2[a] = nd[\"pl\"]\n                    t = term(b2)\n                    if t is not None:\n                        v = -abs(t)\n                    else:\n                        nd[\"ch\"][a] = node(b2, -nd[\"pl\"])\n                        v = nd[\"ch\"][a][\"v\"]\n                    break\n                for n_, a in reversed(path):\n                    v = -v\n                    n_[\"N\"][a] += 1\n                    n_[\"W\"][a] += v\n            pi = root[\"N\"] / root[\"N\"].sum()\n            a = int(rng.choice(9, p=pi)) if sum(1 for x in b if x) < 3 else int(root[\"N\"].argmax())\n            hist.append(pi)\n            b[a] = pl\n            pl = -pl\n        out += hist\n    return out\n\n\ndef check(ns, R_):\n    def t():\n        torch.manual_seed(0)\n        net = ns[\"Net\"]()\n        X = need(ns, \"encode_batch\")([[1, -1, 0, 0, 0, 0, 0, 0, 0], [0] * 9], [-1, 1])\n        ensure(tuple(X.shape) == (2, 18) and X[0, :2].tolist() == [0.0, 1.0] and X[0, 9:11].tolist() == [1.0, 0.0], \"kodlama\")\n        data = need(ns, \"batched_self_play\")(net, 64, 30, np.random.default_rng(0))\n        ensure(5 * 64 <= len(data) <= 9 * 64, \"her oyun 5–9 hamle\")\n        ensure(all(abs(sum(pi) - 1) < 1e-6 and abs(z) <= 1 for _, _, pi, z in data), \"örnekler\")\n        X0 = [d for d in data if not any(d[0])]\n        ensure(len(X0) == 64, \"her oyunun boş tahta örneği\")\n        rows = []\n        for G in (16, 64, 128):\n            tb = bench(lambda: ns[\"batched_self_play\"](net, G, 30, np.random.default_rng(1)), repeat=1, number=1)\n            ts = bench(lambda: seq_self_play(net, min(G, 16), 30, np.random.default_rng(1), ns[\"term\"]), repeat=1, number=1) * G / min(G, 16)\n            rows.append([G, fmt_time(ts), fmt_time(tb), f\"{ts / tb:.1f}×\"])\n        R_.table(\"Kendine karşı oyun, 30 benzetim/hamle\", [\"oyun\", \"sıralı (oyun başına ağ çağrısı)\", \"toplu yaprak değerlendirme\", \"hızlanma\"], rows,\n                 note=\"CPU'da ve küçük ağda kazanç, ağ çağrısı başına sabit maliyetin (Python → C++ → çekirdek) G'ye bölünmesinden gelir; GPU'da ve büyük ağda oran çok daha büyüktür. AlphaGo Zero toplu değerlendirmeyi 8'lik mini-batch'lerle ve sanal kayıpla yapar.\")\n    R_.test(\"batched_self_play\", t)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        rows = []\n        res = {}\n        for n in (8, 16, 64):\n            a = need(ns, \"win_rate\")(False, 200, n, 0)\n            b = ns[\"win_rate\"](True, 200, n, 0)\n            res[n] = (a, b)\n            rows.append([n, f\"{a:.2f}\", f\"{b:.2f}\"])\n        ensure(res[8][1] > res[8][0] + 0.15, \"gürültü kazanan hamleyi az benzetimde bulmayı sağlamalı\")\n        R_.table(\"Önseli 0.001 olan kazanan hamle, 200 deneme\", [\"benzetim\", \"gürültüsüz\", \"Dirichlet(0.3), ε = 0.25\"], rows,\n                 note=\"PUCT'ta ziyaretsiz bir kenarın puanı c·P·√N: P ≈ 0 ise birçok benzetim boyunca seçilmez. Dirichlet(0.3) çoğu kütleyi birkaç rastgele hamleye koyar: her oyunda başka 'şanslı' hamleler denenir. Benzetim çoğaldıkça gürültüsüz arama da sonunda bulur.\")\n    R_.test(\"Dirichlet gürültüsü\", t)\n",
 "x2": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        res = [need(ns, \"experiment\")(150, s) for s in range(3)]\n        a = np.mean([r[0] for r in res]); b = np.mean([r[1] for r in res])\n        ensure(b < 0.85 * a, f\"artırım test hatasını düşürmeli ({a:.3f} → {b:.3f})\")\n        R_.table(\"150 eğitim durumu, ~4370 test durumu (minimax değerleri), 3 tohum\", [\"veri\", \"test MSE\"], [[\"yalnız 150 durum\", f\"{a:.3f}\"], [\"8 simetriyle 1200 örnek\", f\"{b:.3f}\"]],\n                 note=\"Değerin simetrilere değişmez olduğu bilgisi veriye gömülür. AlphaGo Zero artırımı eğitimde, AlphaGo ise ayrıca değerlendirmede (rastgele simetri) kullandı; AlphaZero satranç/shogi için kaldırdı (simetrik değiller).\")\n    R_.test(\"simetri artırımı\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        close(need(ns, \"elo_diff_mle\")(3, 1), 400 * np.log10(3), what=\"MLE\")\n        m, lo, hi = need(ns, \"elo_posterior\")(4, 1)\n        ensure(lo < 0 < m < hi, \"4–1 sonucunda belirsizlik büyük: %90 aralık 0'ı kapsar\")\n        m2, lo2, hi2 = ns[\"elo_posterior\"](100, 0)\n        ensure(lo2 > 500, \"100–0: alt sınır yüksek\")\n        R_.table(\"Maç sonucundan Elo farkı (Beta(1,1) önselli kazanma olasılığı)\", [\"maç\", \"MLE\", \"ortanca\", \"%90 aralık\"],\n                 [[\"AlphaGo Lee – Lee Sedol 4–1\", f\"{ns['elo_diff_mle'](4, 1):.0f}\", f\"{m:.0f}\", f\"[{lo:.0f}, {hi:.0f}]\"],\n                  [\"AlphaGo Zero – AlphaGo Lee 100–0\", \"∞\", f\"{m2:.0f}\", f\"[{lo2:.0f}, {hi2:.0f}]\"]],\n                 note=\"Beş oyun güç farkını kabaca ölçer; 100–0'da MLE sonsuzdur ve yalnız alt sınır anlamlıdır. Makaleler bu yüzden Elo'yu birçok oyuncunun birçok oyunundan Bradley–Terry ile (c aşaması) hesaplar.\")\n    R_.test(\"Elo belirsizliği\", t)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_11_alphazero as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Go:** iki oyuncu sırayla taş koyar; özgürlüğü kalmayan bağlı taşlar alınır; en çok alanı kontrol eden kazanır. Dallanma ~250, derinlik ~150: tam arama imkânsız. AlphaGo (DeepMind) Mart 2016'da Lee Sedol'u (9 dan), Mayıs 2017'de dünya şampiyonu Ke Jie'yi yendi.

**AlphaGo'nun dört aşaması:**

1. **Denetimli politika ağı** $p_\sigma(a|s)$: KGS'den 30 milyon konum; uzman hamlesi $a_i^*$ optimal sayılır, $\max\sum_i\log\pi_\sigma(a_i^*|s_i)$.
2. **RL politika ağı** $p_\rho$: eski sürümlerine karşı oynar; oyun sonu $G_n=\pm1$, γ = 1 ile REINFORCE: $\nabla=\sum_n\nabla\log\pi(a_n|s_n)G_n$.
3. **Değer ağı** $v_\theta(s)$: $p_\rho$'nun kendine karşı oyunlarından $(s,G)$ çiftlerine regresyon, $\tfrac12(G-v_\theta(s))^2$; korelasyonu kırmak için her oyundan bir konum.
4. **Arama:** her kenarda $Q(s,a)$, $P(s,a)$ (= $p_\sigma$), $N(s,a)$; seçim $a^*=\arg\max Q+u$, $u\propto P/(1+N)$; yaprakta $V=(1-\lambda)v_\theta(s_L)+\lambda z_L$ ($z_L$ hızlı rollout sonucu).

**AlphaGo Zero:** insan verisi yok, rollout yok, tek ağ $(p,v)=f_\theta(s)$. Kendine karşı oyunda her hamlede MCTS (800 benzetim); hamle $\pi(a)\propto N(s_0,a)^{1/\tau}$ ile seçilir (ilk 30 hamlede τ = 1, sonra → 0). Kökte $P=(1-\varepsilon)p+\varepsilon\,\mathrm{Dir}(\alpha)$. Oyun sonunda her konuma $z=\pm1$ (sıradaki oyuncunun bakışından). Kayıp:

$$\ell=(z-v)^2-\pi^\top\log p+c\|\theta\|^2.$$

Ağ π'yi taklit eder → daha iyi önsel → daha iyi arama: **arama politika iyileştirme operatörü**, $z$ politika değerlendirme operatörü. Simetriler (8) ile veri artırımı. Elo: $E_A=1/(1+10^{(R_B-R_A)/400})$.

## 1 · Elle

τ = 0.5: $N^2=(100,900,3600)$, $3600/4600$. τ → 0 en çok ziyaret edilene deterministik, τ = 1 orantılı.

Kayıp: $(1-0.5)^2=0.25$, $-\tfrac12\log\tfrac12-\tfrac12\log\tfrac12=\ln2$.

Elo: $1/(1+10^{-0.5})=0.7597$: 200 puan farkı ≈ %76 beklenen skor; AlphaGo Zero'nun AlphaGo Lee'yi 100–0 yenmesi bu ölçekle en az ~800 puana denk gelir (x3).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**AlphaGo**

- `mix`: Yaprak değeri, λ = 0.5: $v_\theta=0.2$, rastgele oyun sonucu $z=1$
- `syms`: Kare tahtanın simetri sayısı (döndürme + yansıma)

**AlphaGo Zero**

- `tau`: N = (10, 30, 60), τ = 0.5: $\pi(a_3)$
- `loss`: Kayıp: z = 1, v = 0.5, π = p = (½, ½), c = 0
- `dir`: Kök önseli: P = 0.4, Dirichlet örneği η = 0.2, ε = 0.25

**Elo**

- `elo`: 200 puan farkı: güçlü oyuncunun beklenen skoru

In [ ]:
cevaplar = dict(
    mix=None,
    syms=None,
    tau=None,
    loss=None,
    dir=None,
    elo=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"mix":[0.6,0],"syms":[8,0],"tau":[0.782608695652174,1e-9],"loss":[0.9431471805599453,1e-9],"dir":[0.35,0],"elo":[0.7597469266479578,1e-9]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**PUCT araması**, saf Python, tic-tac-toe (tahta 9 elemanlı liste; 1 = X, −1 = O). "Ağ" bir fonksiyon: `net_fn(board, player)` → `(prior (9 liste), değer)` — değer **sıradaki oyuncunun** bakışından.

- `terminal(board)` → 1/−1/0/None.
- `puct_search(board, player, net_fn, n_sims, c=1.5)` → kökteki ziyaret sayıları (9 liste). Her düğüm: `P` (yasal hamlelere normalize önsel), `N`, `W` (kenar başına, düğümün oyuncusunun bakışından). Seçim $Q+c\,P\sqrt{\sum N+1}/(1+N)$ (ziyaretsiz kenarda Q = 0). Genişletmede çocuk tahtası terminal ise değer (sıradaki oyuncu için) −1 ya da 0; değilse `net_fn`. Geri yayılımda her seviyede işaret değişir.
- `visit_policy(N, tau)` — $N^{1/\tau}$ normalize; τ = 0 → argmax'a tek-sıcak (eşitlikte ilk).

In [ ]:
%%writefile stage_b.py
import math

LINES = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]


def terminal(board):
    ...


def puct_search(board, player, net_fn, n_sims, c=1.5):
    ...


def visit_policy(N, tau):
    ...


def uniform_net(board, player):
    return [1 / 9] * 9, 0.0


print(puct_search([1, 1, 0, -1, -1, 0, 0, 0, 0], 1, uniform_net, 50))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, döngüsüz (Elo yinelemesi hariç).

- `symmetry_perms()` → `(8, 9)` int: `np.arange(9).reshape(3,3)`'ün 4 döndürmesi ve yansımaları (`np.rot90`, `np.fliplr`); her satır, dönüşmüş tahtanın i. karesinin eski indeksi.
- `augment(X, PI, Z)` — `X (N, 18)` (iki 9'luk düzlem), `PI (N, 9)`, `Z (N,)` → 8N örnek (gelişmiş indeksleme).
- `az_loss(z, v, pi, logits, theta_sq, c=1e-4)` — ortalama $(z-v)^2-\sum\pi\log\mathrm{softmax}(\text{logits})+c\,\|\theta\|^2$ (logsumexp kararlı).
- `elo_expected(ra, rb)`; `fit_elo(Wm, n_iter=500)` — `Wm[i, j]` i'nin j'ye karşı skoru (beraberlik ½); Bradley–Terry MM (Hunter 2004): $\gamma_i\leftarrow W_i/\sum_{j\ne i}n_{ij}/(\gamma_i+\gamma_j)$ (köşegen yok sayılır); derece $400\log_{10}\gamma$, ortalaması 0.

In [ ]:
%%writefile stage_c.py
import numpy as np


def symmetry_perms():
    ...


def augment(X, PI, Z):
    ...


def az_loss(z, v, pi, logits, theta_sq, c=1e-4):
    ...


def elo_expected(ra, rb):
    ...


def fit_elo(Wm, n_iter=500):
    ...


print(symmetry_perms().shape)

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

**Mini AlphaZero**, tic-tac-toe, PyTorch.

- `class Net` — 18 → 64 → 64 (ReLU), politika logit'leri (9) ve $\tanh$ değer. `encode(board, player)` → 18 (sıradaki oyuncunun taşları, rakibinkiler).
- `mcts(net, board, player, n_sims, rng, c=1.5, noise=True)` → kök ziyaret sayıları (NumPy). Kökte Dirichlet(0.3), ε = 0.25. Düğüm oluşturulurken ağ değerlendirilir (önsel yasal hamlelere maskelenir).
- `self_play(net, n_sims, rng)` → `[(board, player, π, z), …]`: ilk 3 hamlede π'den örnekle, sonra argmax.
- `train(net, opt, data, epochs=3, bs=64)` — 8 simetriyle artır, kayıp $(z-v)^2-\pi^\top\log p$ (L2 `weight_decay` ile).
- `alphazero(iters=15, games=20, n_sims=30, seed=0)` → ağ: her yinelemede `games` oyun, son 3000 örnek tamponu, Adam(2e-3, wd 1e-4).

In [ ]:
%%writefile stage_d.py
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

LINES = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]


def term(b):
    for l in LINES:
        s = b[l[0]] + b[l[1]] + b[l[2]]
        if s == 3 or s == -3:
            return s // 3
    return 0 if all(b) else None


class Net(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        ...

    def forward(self, x):
        ...


def encode(board, player):
    ...


def mcts(net, board, player, n_sims, rng, c=1.5, noise=True):
    ...


def self_play(net, n_sims, rng):
    ...


def train(net, opt, data, epochs=3, bs=64):
    ...


def alphazero(iters=15, games=20, n_sims=30, seed=0):
    ...


net = alphazero(2, 5, 10)
print(mcts(net, [0] * 9, 1, 30, np.random.default_rng(0), noise=False))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

**Toplu kendine karşı oyun:** G oyun aynı anda; her benzetim adımında her ağaçta seçimle bir yaprağa in, tüm yaprakları **tek** ileri geçişte değerlendir, sonra genişlet ve geri yay. Python ağaç işlemleri sürer, ama ağ çağrısı G kat azalır (GPU'da asıl kazanç budur).

- `encode_batch(boards, players)` → `(n, 18)`.
- `batched_self_play(net, n_games, n_sims, rng, c=1.5)` → d'deki `self_play` ile aynı biçimde örnekler (tüm oyunlar birleşik).

In [ ]:
%%writefile stage_e.py
import math
import numpy as np
import torch
import torch.nn as nn

LINES = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]


def term(b):
    for l in LINES:
        s = b[l[0]] + b[l[1]] + b[l[2]]
        if s == 3 or s == -3:
            return s // 3
    return 0 if all(b) else None


class Net(nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(18, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU())
        self.p = nn.Linear(h, 9)
        self.v = nn.Linear(h, 1)

    def forward(self, x):
        z = self.body(x)
        return self.p(z), torch.tanh(self.v(z)).squeeze(-1)


def encode_batch(boards, players):
    ...


def batched_self_play(net, n_games, n_sims, rng, c=1.5):
    ...


torch.manual_seed(0)
print(len(batched_self_play(Net(), 8, 20, np.random.default_rng(0))))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Kökte Dirichlet gürültüsü neden gerekli?

Kazanan hamlenin önseli neredeyse sıfır olan bir konum: X, 2'ye oynarsa kazanır; ağ önseli 2'ye 0.001, diğer yasal hamlelere eşit; değer her zaman 0.

- `root_visits(n_sims, noise, rng)` — b'deki PUCT'u (kendi kopyan) kökte isteğe bağlı Dirichlet(0.3), ε = 0.25 ile çalıştır; kazanan hamlenin ziyaret oranı.
- `win_rate(noise, n_trials=200, n_sims=8, seed=0)` — en çok ziyaret edilen hamlenin kazanan hamle olma oranı.

In [ ]:
%%writefile extra_x1.py
import math
import numpy as np

LINES = [[0, 1, 2], [3, 4, 5], [6, 7, 8], [0, 3, 6], [1, 4, 7], [2, 5, 8], [0, 4, 8], [2, 4, 6]]
BOARD = [1, 0, 0, -1, 1, 0, 0, -1, 0]  # X'in sırası; 8'e oynarsa (çapraz 0-4-8) kazanır
WIN = 8


def root_visits(n_sims, noise, rng):
    ...


def win_rate(noise, n_trials=200, n_sims=8, seed=0):
    ...


print(win_rate(False), win_rate(True))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Simetri artırımı ile genelleme

Tüm 4520 terminal olmayan tic-tac-toe durumunun kusursuz değerleri (minimax) verilir. Küçük bir MLP (18 → 32 → 1, tanh, NumPy, tam batch gradyan inişi) yalnız **150 durumla** eğitilir.

- `train_value(X, y, iters=3000, lr=0.05, seed=0)` → ağırlıklar; `predict(params, X)`.
- `experiment(n_train=150, seed=0)` → `(test MSE artırımsız, test MSE 8 simetriyle artırılmış)`.

In [ ]:
%%writefile extra_x2.py
import numpy as np


def train_value(X, y, iters=3000, lr=0.05, seed=0):
    ...


def predict(params, X):
    ...


def experiment(n_train=150, seed=0):
    ...


print(experiment())

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Maç sonuçlarından Elo farkı ve belirsizliği

- `elo_diff_mle(wins, losses)` → $400\log_{10}(w/l)$ (beraberliksiz).
- `elo_posterior(wins, losses, n_samples=200000, seed=0)` — Beta(1 + w, 1 + l) sonsalından kazanma olasılığı örnekle, Elo farkına çevir → `(ortanca, %5, %95)`.

In [ ]:
%%writefile extra_x3.py
import numpy as np


def elo_diff_mle(wins, losses):
    ...


def elo_posterior(wins, losses, n_samples=200000, seed=0):
    ...


print(elo_diff_mle(4, 1), elo_posterior(4, 1))

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

## Makale ödevi: Silver ve ark. (2016): AlphaGo'nun boru hattı

[Silver ve ark. — Mastering the game of Go with deep neural networks and tree search (Nature 2016)](https://doi.org/10.1038/nature16961)

AlphaGo'nun dört aşamasını küçük bir oyunda (9×9 Go yerine 6×6 Othello ya da Connect-4) yeniden kur: (1) güçlü bir arama motorunun (ör. derin MCTS) oyunlarından SL politika ağı, (2) havuzdan rastgele seçilen eski sürümlere karşı REINFORCE ile RL politika ağı, (3) her oyundan tek konum örnekleyerek değer ağı, (4) PUCT araması: önsel = SL politikası, yaprak = $(1-\lambda)v_\theta+\lambda z$ (hızlı rollout politikasıyla). λ ∈ {0, 0.5, 1} ve "önsel olarak RL politikası" ablation'larını yap.


**Adımlar**

1. Makalenin Şekil 4'ündeki gibi bileşen ablation tablosu: Elo'yu c aşamasındaki Bradley–Terry ile hesapla.
2. Her oyundan tek konum vs tüm konumlar ile değer ağı test MSE'si.

**Kabul ölçütleri**

- λ = 0.5 karışımı tek başına değer ağı ya da tek başına rollout'tan güçlü (makale Şekil 4b nitel).
- SL politikası önsel olarak RL politikasından iyi (makalenin şaşırtıcı bulgusu: daha çeşitli).

## Makale ödevi: Silver ve ark. (2017): AlphaGo Zero

[Silver ve ark. — Mastering the game of Go without human knowledge (Nature 2017)](https://doi.org/10.1038/nature24270)

d ve e aşamalarındaki mini AlphaZero'yu Connect-4'e (6×7) taşı: artık ağ (ResNet, politika ve değer başlıkları), toplu kendine karşı oyun, değerlendirici (yeni ağ eskisini %55 ile yenerse kabul; AlphaZero'da bu adım yok — ikisini karşılaştır), 8 yerine 2 simetri (yalnız yatay yansıma). Elo eğrisini eğitim süresine karşı çiz.


**Adımlar**

1. Bilinen çözüm: ilk oyuncu orta sütundan başlarsa kazanır — ağın değer başlığının boş tahtada pozitif olduğunu doğrula.
2. Ablation: tek ağ vs ayrı politika/değer ağları (makale Şekil 4).

**Kabul ölçütleri**

- Elo düzenli artar; birleşik ağ ayrı ağlardan iyi (makale Şekil 4 nitel).
- Son ağ + 800 benzetim, d aşamasındaki rastgele-oyunlu UCT'yi ≥ %90 yener.

## Kaynaklar

- [Silver ve ark. — Mastering the game of Go with deep neural networks and tree search (Nature 2016)](https://doi.org/10.1038/nature16961) — ©
- [Silver ve ark. — Mastering the game of Go without human knowledge (Nature 2017)](https://doi.org/10.1038/nature24270) — ©
- [Silver ve ark. — A general reinforcement learning algorithm that masters chess, shogi, and Go through self-play (Science 2018)](https://doi.org/10.1126/science.aar6404) — ©
- [Hunter — MM algorithms for generalized Bradley–Terry models (Ann. Stat. 2004)](https://doi.org/10.1214/aos/1079120141) — ©
- [Anthony, Tian, Barber — Thinking Fast and Slow with Deep Learning and Tree Search (NeurIPS 2017)](https://arxiv.org/abs/1705.08439) — arXiv